# 02 — LLM-as-Policy: Dual-Agent Zero-Shot Battery Control

**Phase 1 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

This notebook evaluates mid-cost LLMs as zero-shot battery controllers in a **two-agent setup**
that mirrors the Phase 4 multi-agent deployment:

| Agent | Buildings | Observability |
|-------|-----------|---------------|
| **α** | B0, B1, B2 | Sees only its own 3 buildings |
| **β** | B3, B4, B5 | Sees only its own 3 buildings |

Each timestep makes **2 independent LLM calls** (one per agent). The district reward is the
sum across all 6 buildings. Neither agent knows what the other is doing — this is the
Action-Only coordination baseline.

**State includes short-horizon forecasts:** price +6 h / +12 h (LOW/PEAK) and solar irradiance
+6 h (NONE/LOW/HIGH) so the LLM can plan charge/discharge timing across the next half-day.

**Resilience design:** every provider runs in its **own separate cell**. If a call hangs,
interrupt that cell — other providers are unaffected. A per-call `LLM_TIMEOUT_S` guard
returns fallback zeros and continues the rollout rather than blocking forever.

**Providers tested:**

| Provider | Model | API endpoint |
|----------|-------|-------------|
| Anthropic | `claude-haiku-4-5` | Native |
| DeepSeek | `deepseek-chat` (V3) | OpenAI-compat |
| Kimi | `kimi-k2.5` | OpenAI-compat |
| NVIDIA NIM | `meta/llama-3.1-8b-instruct` | OpenAI-compat |
| Google Gemma | `gemma-3-12b-it` | OpenAI-compat (Google AI Studio) |

**API keys:** loaded from `.env` at project root.  
Required vars: `ANTHROPIC_API_KEY`, `DEEPSEEK_API_KEY`, `KIMI_API_KEY`, `NVIDIA_API_KEY`, `GOOGLE_API_KEY`.

## § 0 — Config
> **Change experiment parameters here only.** Nothing else needs editing.

In [13]:
import sys
from pathlib import Path

# ── Make src/ importable ──────────────────────────────────────────────────
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# ── Provider list ─────────────────────────────────────────────────────────
# Comment / uncomment entries to control which providers run.
# Each dict is passed as kwargs to LLMProvider().
PROVIDERS: list[dict] = [
    {
        "name":     "anthropic",
        "model":    "claude-haiku-4-5",
        "key_env":  "ANTHROPIC_API_KEY",
        "base_url": None,
    },
    {
        "name":     "deepseek",
        "model":    "deepseek-chat",
        "key_env":  "DEEPSEEK_API_KEY",
        "base_url": "https://api.deepseek.com/v1",
    },
    {
        "name":     "kimi",
        "model":    "kimi-k2.5",
        "key_env":  "KIMI_API_KEY",
        "base_url": "https://api.moonshot.ai/v1",
    },
    {
        "name":     "nvidia",
        "model":    "meta/llama-3.1-8b-instruct",
        "key_env":  "NVIDIA_API_KEY",
        "base_url": "https://integrate.api.nvidia.com/v1",
    },
    {
        "name":     "gemma",
        "model":    "gemma-3-12b-it",
        "key_env":  "GOOGLE_API_KEY",
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
    },
]

# ── Dual-agent split ─────────────────────────────────────────────────────
AGENT_A_BUILDINGS: list[int] = [0, 1, 2]   # α manages these env building indices
AGENT_B_BUILDINGS: list[int] = [3, 4, 5]   # β manages these env building indices

# ── Timeout ──────────────────────────────────────────────────────────────
# Seconds per LLM call before returning fallback zeros and continuing.
# Timeout breaks immediately — no retry — so a single hung call wastes at
# most LLM_TIMEOUT_S seconds instead of freezing the rollout.
LLM_TIMEOUT_S: float = 45.0

# ── Experiment window ────────────────────────────────────────────────────
WEEK_START: int = 3624   # mid-December; matches 04_llm_policy_clean for comparison
WEEK_LEN:   int = 168    # one week (168 hourly steps)

# ── Output ───────────────────────────────────────────────────────────────
ARTIFACTS = Path("artifacts").resolve()
ARTIFACTS.mkdir(exist_ok=True)

# ── Sanity print ─────────────────────────────────────────────────────────
import os
print(f"Project root   : {PROJECT_ROOT}")
print(f"Window         : t{WEEK_START}..{WEEK_START + WEEK_LEN - 1} ({WEEK_LEN} steps)")
print(f"Agent α        : buildings {AGENT_A_BUILDINGS}")
print(f"Agent β        : buildings {AGENT_B_BUILDINGS}")
print(f"Timeout/call   : {LLM_TIMEOUT_S}s")
print(f"\nProviders ({len(PROVIDERS)}):")
for p in PROVIDERS:
    has_key = bool(os.environ.get(p["key_env"], "").strip())
    flag = "✓" if has_key else "✗ MISSING"
    print(f"  [{flag}] {p['name']:10s} model={p['model']:35s} env={p['key_env']}")

Project root   : /Users/antonisbast/projects/eclipse-thesis
Window         : t3624..3791 (168 steps)
Agent α        : buildings [0, 1, 2]
Agent β        : buildings [3, 4, 5]
Timeout/call   : 45.0s

Providers (5):
  [✓] anthropic  model=claude-haiku-4-5                    env=ANTHROPIC_API_KEY
  [✓] deepseek   model=deepseek-chat                       env=DEEPSEEK_API_KEY
  [✓] kimi       model=kimi-k2.5                           env=KIMI_API_KEY
  [✓] nvidia     model=meta/llama-3.1-8b-instruct          env=NVIDIA_API_KEY
  [✓] gemma      model=gemma-3-12b-it                      env=GOOGLE_API_KEY


## § 1 — Imports

In [14]:
import json
import random
import time
import warnings

import citylearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from citylearn.citylearn import CityLearnEnv

# ── src/ modules ──────────────────────────────────────────────────────────
from src.env import SEED, BUILDINGS, make_env, snapshot_state
from src.agent import (
    LLMProvider, make_policy_llm, make_system_prompt,
    render_state, parse_actions,
    price_bucket, carbon_bucket, solar_bucket,
    PRICE_PEAK_THRESHOLD,
)

warnings.filterwarnings("ignore")
np.random.seed(SEED)
random.seed(SEED)

print(f"CityLearn {citylearn.__version__}")
print("src.env and src.agent loaded.")

CityLearn 2.5.0
src.env and src.agent loaded.


## § 2 — Environment

In [ ]:
_env = make_env(
    start=WEEK_START,
    end=WEEK_START + WEEK_LEN - 1,
    obs_set="llm",
    reward_fn="merlin",
)
_obs, _ = _env.reset()
print(f"buildings      : {len(_obs)}")
print(f"obs / building : {len(_obs[0])}  (llm obs set — 9 real-time vars in the obs vector)")
print(f"steps / episode: {_env.time_steps}")
print(f"reward fn      : {type(_env.reward_function).__name__}")

_snap = snapshot_state(_env)
_FORECAST_KEYS = [
    "electricity_pricing_predicted_1",
    "electricity_pricing_predicted_2",
    "solar_irradiance_predicted_1",
]

print(f"\nsnapshot_state() returns {len(_snap[0])} fields per building:")
print(f"  9 real-time : month, day_type, hour, electricity_pricing, carbon_intensity,")
print(f"                solar_generation, non_shiftable_load, electrical_storage_soc,")
print(f"                net_electricity_consumption_last")
print(f"  3 forecasts : electricity_pricing_predicted_1  (price  +6 h, $/kWh)")
print(f"                electricity_pricing_predicted_2  (price +12 h, $/kWh)")
print(f"                solar_irradiance_predicted_1     (irradiance +6 h, W/m²)")

print(f"\nForecast availability at t=0 (None → column missing in dataset):")
d0 = _snap[0]
for k in _FORECAST_KEYS:
    print(f"  {k}: {d0[k]}")

print(f"\nBuilding states at t=0:")
for i, d in enumerate(_snap):
    print(
        f"  B{i}: SoC={d['electrical_storage_soc']*100:.1f}%  "
        f"load={d['non_shiftable_load']:.2f}  "
        f"price={d['electricity_pricing']:.3f}  "
        f"price+6h={d['electricity_pricing_predicted_1']}  "
        f"irr+6h={d['solar_irradiance_predicted_1']}"
    )

## § 3 — State Renderer

`snapshot_state()` returns **12 fields** per building: 9 real-time signals + 3 short-horizon
forecasts (price +6 h, price +12 h, solar irradiance +6 h W/m²).

`render_state()` converts these into a human-readable prompt string, inserting a **Forecast**
line between the district header and the per-building table, e.g.:

```
Month 12, Wed 08:00  |  price=0.210 (LOW)  |  carbon=0.202 (PEAK)
Forecast:  price+6h=PEAK  price+12h=LOW  solar+6h=HIGH
Buildings:
  B0: SoC= 45.3%  load=1.20 kWh  last_net=+0.85 kWh  solar=LOW
  ...
```

Each agent receives only its **own 3 buildings**. `render_state()` re-labels from B0
regardless of the slice, so both agents see identically structured prompts.

In [16]:
snap_a = _snap[0:3]   # Agent α view
snap_b = _snap[3:6]   # Agent β view

print("=" * 60)
print("Agent α state (buildings 0-2):")
print("=" * 60)
print(render_state(snap_a))

print()
print("=" * 60)
print("Agent β state (buildings 3-5):")
print("=" * 60)
print(render_state(snap_b))

Agent α state (buildings 0-2):
Month 12, Wed 24:00  |  price=0.210 (LOW)  |  carbon=0.202 (PEAK)
Buildings:
  B0: SoC=  0.0%  load=0.60 kWh  last_net=+0.00 kWh  solar=NONE
  B1: SoC=  0.0%  load=1.68 kWh  last_net=+0.00 kWh  solar=NONE
  B2: SoC=  0.0%  load=0.28 kWh  last_net=+0.00 kWh  solar=NONE

Agent β state (buildings 3-5):
Month 12, Wed 24:00  |  price=0.210 (LOW)  |  carbon=0.202 (PEAK)
Buildings:
  B0: SoC=  0.0%  load=0.45 kWh  last_net=+0.00 kWh  solar=NONE
  B1: SoC=  0.0%  load=1.03 kWh  last_net=+0.00 kWh  solar=NONE
  B2: SoC=  0.0%  load=1.59 kWh  last_net=+0.00 kWh  solar=NONE


## § 4 — System Prompts

`make_system_prompt(n)` generates the controller prompt for `n` buildings — action-format
block and peak-demand estimates scale automatically. Both agents use `n=3`.

In [17]:
PROMPT_3B = make_system_prompt(3)   # used by both α and β
PROMPT_6B = make_system_prompt(6)   # kept for single-agent reference baselines

print("3-building prompt (tail — action format):")
print("-" * 50)
# Show just the output-format block
for line in PROMPT_3B.splitlines()[-6:]:
    print(line)

3-building prompt (tail — action format):
--------------------------------------------------
Step 4: Output exactly 3 action lines.

OUTPUT FORMAT (strict — nothing after the last </action> tag):
<action building=0>VALUE</action>
<action building=1>VALUE</action>
<action building=2>VALUE</action>


## § 5 — Provider Setup & Smoke Test

`PROVIDER_OBJS` is initialised in the first cell below. Each subsequent cell smoke-tests
**one provider independently** with a short `Say PONG.` call — you can safely interrupt a
hung cell without losing providers that already passed.

Providers that fail (timeout, missing API key, import error) are excluded from `PROVIDER_OBJS`
and automatically skipped by every rollout cell in §§ 9–13.

In [ ]:
# Initialise the provider registry — run once before the per-provider cells below.
# Re-running this cell wipes any previously initialised providers.
PROVIDER_OBJS: dict[str, LLMProvider] = {}
print("PROVIDER_OBJS initialised — run per-provider smoke-test cells below.")
print(f"Providers configured in § 0: {[p['name'] for p in PROVIDERS]}")

In [ ]:
# ── Smoke test: Anthropic ─────────────────────────────────────────────────
# Safe to interrupt — only this provider is affected.
_spec = next((s for s in PROVIDERS if s["name"] == "anthropic"), None)
if _spec is None:
    print("[anthropic] not in PROVIDERS list — skipped")
else:
    try:
        _p   = LLMProvider(**_spec)
        _out = _p.complete("You answer in one word.", "Say PONG.", timeout_s=20.0)
        PROVIDER_OBJS["anthropic"] = _p
        print(f"  [✓] {_p.label:45s} → {_out.strip()[:40]!r}")
    except Exception as _exc:
        print(f"  [✗] anthropic  {_spec['model']:35s} skipped: {_exc}")
print(f"Active providers: {list(PROVIDER_OBJS.keys())}")

In [ ]:
# ── Smoke test: DeepSeek ─────────────────────────────────────────────────
# Safe to interrupt — only this provider is affected.
_spec = next((s for s in PROVIDERS if s["name"] == "deepseek"), None)
if _spec is None:
    print("[deepseek] not in PROVIDERS list — skipped")
else:
    try:
        _p   = LLMProvider(**_spec)
        _out = _p.complete("You answer in one word.", "Say PONG.", timeout_s=20.0)
        PROVIDER_OBJS["deepseek"] = _p
        print(f"  [✓] {_p.label:45s} → {_out.strip()[:40]!r}")
    except Exception as _exc:
        print(f"  [✗] deepseek   {_spec['model']:35s} skipped: {_exc}")
print(f"Active providers: {list(PROVIDER_OBJS.keys())}")

In [ ]:
# ── Smoke test: Kimi / Moonshot ───────────────────────────────────────────
# Safe to interrupt — only this provider is affected.
_spec = next((s for s in PROVIDERS if s["name"] == "kimi"), None)
if _spec is None:
    print("[kimi] not in PROVIDERS list — skipped")
else:
    try:
        _p   = LLMProvider(**_spec)
        _out = _p.complete("You answer in one word.", "Say PONG.", timeout_s=20.0)
        PROVIDER_OBJS["kimi"] = _p
        print(f"  [✓] {_p.label:45s} → {_out.strip()[:40]!r}")
    except Exception as _exc:
        print(f"  [✗] kimi       {_spec['model']:35s} skipped: {_exc}")
print(f"Active providers: {list(PROVIDER_OBJS.keys())}")

In [ ]:
# ── Smoke test: NVIDIA NIM ────────────────────────────────────────────────
# Safe to interrupt — only this provider is affected.
# NIM can be slow to cold-start; timeout_s=30 gives it a bit more headroom.
_spec = next((s for s in PROVIDERS if s["name"] == "nvidia"), None)
if _spec is None:
    print("[nvidia] not in PROVIDERS list — skipped")
else:
    try:
        _p   = LLMProvider(**_spec)
        _out = _p.complete("You answer in one word.", "Say PONG.", timeout_s=30.0)
        PROVIDER_OBJS["nvidia"] = _p
        print(f"  [✓] {_p.label:45s} → {_out.strip()[:40]!r}")
    except Exception as _exc:
        print(f"  [✗] nvidia     {_spec['model']:35s} skipped: {_exc}")
print(f"Active providers: {list(PROVIDER_OBJS.keys())}")

In [ ]:
# ── Smoke test: Google Gemma ──────────────────────────────────────────────
# Safe to interrupt — only this provider is affected.
_spec = next((s for s in PROVIDERS if s["name"] == "gemma"), None)
if _spec is None:
    print("[gemma] not in PROVIDERS list — skipped")
else:
    try:
        _p   = LLMProvider(**_spec)
        _out = _p.complete("You answer in one word.", "Say PONG.", timeout_s=20.0)
        PROVIDER_OBJS["gemma"] = _p
        print(f"  [✓] {_p.label:45s} → {_out.strip()[:40]!r}")
    except Exception as _exc:
        print(f"  [✗] gemma      {_spec['model']:35s} skipped: {_exc}")
print(f"Active providers: {list(PROVIDER_OBJS.keys())}")

## § 6 — Rollout Functions

In [ ]:
def run_policy(
    name: str,
    policy_fn,
    start: int = WEEK_START,
    length: int = WEEK_LEN,
    reward_fn: str = "merlin",
) -> tuple[pd.DataFrame, CityLearnEnv, list[dict]]:
    """Single-agent rollout — policy_fn sees all 6 buildings.

    policy_fn signature: (snap, t) -> list[float]  OR  (actions, raw, fallback).
    """
    env = make_env(start=start, end=start + length - 1, obs_set="llm", reward_fn=reward_fn)
    env.reset()

    rows, raw_log = [], []
    done, t, t0 = False, 0, time.time()

    while not done:
        snap   = snapshot_state(env)
        result = policy_fn(snap, t)

        if isinstance(result, tuple):
            acts, raw, fb = result
            raw_log.append({"t": t, "state_text": render_state(snap), "raw": raw, "fallback": bool(fb)})
        else:
            acts = result

        _obs, reward, terminated, truncated, _ = env.step([[float(a)] for a in acts])
        done = bool(terminated or truncated)
        post = snapshot_state(env)

        rows.append({
            "policy": name, "t": t, "price": snap[0]["electricity_pricing"],
            "reward_sum": float(np.sum(reward)),
            **{f"a{i}":   acts[i]                                      for i in range(6)},
            **{f"r{i}":   float(reward[i])                             for i in range(6)},
            **{f"soc{i}": post[i]["electrical_storage_soc"]            for i in range(6)},
            **{f"net{i}": post[i]["net_electricity_consumption_last"]  for i in range(6)},
        })
        t += 1

    df = pd.DataFrame(rows)
    n_fb  = sum(1 for r in raw_log if r["fallback"])
    fb_msg = f" | fallbacks={n_fb}/{len(raw_log)}" if raw_log else ""
    print(f"[{name}] {t} steps in {time.time()-t0:.1f}s | reward={df['reward_sum'].sum():.4f}{fb_msg}")
    return df, env, raw_log

In [ ]:
def run_policy_dual_agent(
    name: str,
    policy_a,
    policy_b,
    agent_a_bldgs: list[int] = AGENT_A_BUILDINGS,
    agent_b_bldgs: list[int] = AGENT_B_BUILDINGS,
    start: int = WEEK_START,
    length: int = WEEK_LEN,
    reward_fn: str = "merlin",
) -> dict:
    """Dual-agent rollout: two LLM calls per step, partial observability.

    Agent α receives snap[agent_a_bldgs], Agent β receives snap[agent_b_bldgs].
    Their actions are concatenated (α first, β second) before env.step().

    Returns a dict with keys: df, env, raw_log_a, raw_log_b.
    """
    n_a = len(agent_a_bldgs)
    n_b = len(agent_b_bldgs)

    env = make_env(start=start, end=start + length - 1, obs_set="llm", reward_fn=reward_fn)
    env.reset()

    rows, raw_log_a, raw_log_b = [], [], []
    done, t, t0 = False, 0, time.time()

    while not done:
        snap   = snapshot_state(env)
        snap_a = [snap[i] for i in agent_a_bldgs]   # α sees only its buildings
        snap_b = [snap[i] for i in agent_b_bldgs]   # β sees only its buildings

        # ── Two independent LLM calls ─────────────────────────────────────
        result_a = policy_a(snap_a, t)
        result_b = policy_b(snap_b, t)

        if isinstance(result_a, tuple):
            acts_a, raw_a, fb_a = result_a
            raw_log_a.append({"t": t, "state_text": render_state(snap_a), "raw": raw_a, "fallback": bool(fb_a)})
        else:
            acts_a, fb_a = result_a, False

        if isinstance(result_b, tuple):
            acts_b, raw_b, fb_b = result_b
            raw_log_b.append({"t": t, "state_text": render_state(snap_b), "raw": raw_b, "fallback": bool(fb_b)})
        else:
            acts_b, fb_b = result_b, False

        # ── Combine α + β into a single 6-action vector ───────────────────
        # Reconstruct in original building order
        acts_combined = [0.0] * (n_a + n_b)
        for local_i, global_i in enumerate(agent_a_bldgs):
            acts_combined[global_i] = acts_a[local_i]
        for local_i, global_i in enumerate(agent_b_bldgs):
            acts_combined[global_i] = acts_b[local_i]

        _obs, reward, terminated, truncated, _ = env.step([[float(a)] for a in acts_combined])
        done = bool(terminated or truncated)
        post = snapshot_state(env)

        rows.append({
            "policy": name, "t": t, "price": snap[0]["electricity_pricing"],
            "reward_sum":  float(np.sum(reward)),
            "reward_a":    float(sum(reward[i] for i in agent_a_bldgs)),
            "reward_b":    float(sum(reward[i] for i in agent_b_bldgs)),
            "fallback_a":  fb_a,
            "fallback_b":  fb_b,
            **{f"a{i}":   acts_combined[i]                             for i in range(6)},
            **{f"r{i}":   float(reward[i])                             for i in range(6)},
            **{f"soc{i}": post[i]["electrical_storage_soc"]            for i in range(6)},
            **{f"net{i}": post[i]["net_electricity_consumption_last"]  for i in range(6)},
        })
        t += 1

    df  = pd.DataFrame(rows)
    n_fb_a = sum(1 for r in raw_log_a if r["fallback"])
    n_fb_b = sum(1 for r in raw_log_b if r["fallback"])
    print(
        f"[{name}] {t} steps in {time.time()-t0:.1f}s | "
        f"reward={df['reward_sum'].sum():.4f} "
        f"(α={df['reward_a'].sum():.4f}  β={df['reward_b'].sum():.4f}) | "
        f"fallbacks α={n_fb_a} β={n_fb_b}"
    )
    return {"df": df, "env": env, "raw_log_a": raw_log_a, "raw_log_b": raw_log_b}

## § 7 — Reference Policies

In [ ]:
def policy_noop(snap: list[dict], t: int) -> list[float]:
    return [0.0] * len(snap)

_rng = np.random.default_rng(SEED)

def policy_random(snap: list[dict], t: int) -> list[float]:
    return _rng.uniform(-1.0, 1.0, size=len(snap)).tolist()

def policy_rbc(snap: list[dict], t: int) -> list[float]:
    """Price + solar aware RBC exploiting the battery asymmetry insight."""
    acts = []
    for d in snap:
        soc = d["electrical_storage_soc"]
        prc = d["electricity_pricing"]
        sol = d["solar_generation"]
        if solar_bucket(sol) == "HIGH" and soc < 0.85:
            acts.append(0.2)
        elif price_bucket(prc) == "PEAK" and soc > 0.10:
            acts.append(-1.0)
        elif price_bucket(prc) == "LOW" and soc < 0.90:
            acts.append(0.25)
        else:
            acts.append(0.0)
    return acts

print("Reference policies defined: noop, random, rbc.")

Reference policies defined: noop, random, rbc.


## § 8 — Reference Baselines

In [ ]:
df_noop,   env_noop,   _ = run_policy("noop",   policy_noop)
df_random, env_random, _ = run_policy("random", policy_random)
df_rbc,    env_rbc,    _ = run_policy("rbc",    policy_rbc)

[noop] 167 steps in 0.3s | reward=-1282.7591
[random] 167 steps in 0.3s | reward=-3084.5788
[rbc] 167 steps in 0.3s | reward=-2449.3846


## § 8b — Initialise Results Store

Run this cell **once** before any provider cell. Each provider cell below adds one entry to
`llm_runs`. Re-running this cell wipes all LLM results — use it to start fresh.

In [ ]:
llm_runs: dict[str, dict] = {}
print("llm_runs initialised — ready for provider cells.")

llm_runs initialised — ready for provider cells.


## § 9 — Anthropic: `claude-haiku-4-5`

> **Independent cell.** Interrupt safely if it hangs — other providers are unaffected.  
> Two calls per step (α + β) × 168 steps ≈ 336 API calls. Estimated cost: **~$0.30**.

In [ ]:
_name = "anthropic"

if _name not in PROVIDER_OBJS:
    print(f"[{_name}] not available — skipped (see § 5 smoke test)")
else:
    _p = PROVIDER_OBJS[_name]
    _policy_a = make_policy_llm(_p, n_buildings=3, agent_label="α", timeout_s=LLM_TIMEOUT_S)
    _policy_b = make_policy_llm(_p, n_buildings=3, agent_label="β", timeout_s=LLM_TIMEOUT_S)

    print(f"=== Dual-agent: {_p.label} | timeout={LLM_TIMEOUT_S}s ===")
    _t0  = time.time()
    _run = run_policy_dual_agent(f"llm_{_name}", _policy_a, _policy_b)
    print(f"  wall clock: {time.time()-_t0:.0f}s")
    llm_runs[_name] = {"label": _p.label, **_run}
    print(f"  saved to llm_runs['{_name}']. Active runs: {list(llm_runs.keys())}")

=== Dual-agent: anthropic:claude-haiku-4-5 | timeout=45.0s ===


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


KeyboardInterrupt: 

## § 10 — DeepSeek: `deepseek-chat`

> **Independent cell.** Interrupt safely if it hangs.  
> Estimated cost: **~$0.05**.

In [ ]:
_name = "deepseek"

if _name not in PROVIDER_OBJS:
    print(f"[{_name}] not available — skipped (see § 5 smoke test)")
else:
    _p = PROVIDER_OBJS[_name]
    _policy_a = make_policy_llm(_p, n_buildings=3, agent_label="α", timeout_s=LLM_TIMEOUT_S)
    _policy_b = make_policy_llm(_p, n_buildings=3, agent_label="β", timeout_s=LLM_TIMEOUT_S)

    print(f"=== Dual-agent: {_p.label} | timeout={LLM_TIMEOUT_S}s ===")
    _t0  = time.time()
    _run = run_policy_dual_agent(f"llm_{_name}", _policy_a, _policy_b)
    print(f"  wall clock: {time.time()-_t0:.0f}s")
    llm_runs[_name] = {"label": _p.label, **_run}
    print(f"  saved to llm_runs['{_name}']. Active runs: {list(llm_runs.keys())}")

## § 11 — Kimi: `kimi-k2.5`

> **Independent cell.** Interrupt safely if it hangs.  
> Estimated cost: **~$0.08**.

In [ ]:
_name = "kimi"

if _name not in PROVIDER_OBJS:
    print(f"[{_name}] not available — skipped (see § 5 smoke test)")
else:
    _p = PROVIDER_OBJS[_name]
    _policy_a = make_policy_llm(_p, n_buildings=3, agent_label="α", timeout_s=LLM_TIMEOUT_S)
    _policy_b = make_policy_llm(_p, n_buildings=3, agent_label="β", timeout_s=LLM_TIMEOUT_S)

    print(f"=== Dual-agent: {_p.label} | timeout={LLM_TIMEOUT_S}s ===")
    _t0  = time.time()
    _run = run_policy_dual_agent(f"llm_{_name}", _policy_a, _policy_b)
    print(f"  wall clock: {time.time()-_t0:.0f}s")
    llm_runs[_name] = {"label": _p.label, **_run}
    print(f"  saved to llm_runs['{_name}']. Active runs: {list(llm_runs.keys())}")

## § 12 — NVIDIA NIM: `meta/llama-3.1-8b-instruct`

> **Independent cell.** Interrupt safely if it hangs.  
> NIM uses an OpenAI-compatible endpoint (`https://integrate.api.nvidia.com/v1`).  
> To swap models, edit `PROVIDERS` in § 0 — e.g. `meta/llama-3.3-70b-instruct` for higher quality.  
> Estimated cost: **~$0.05** (8B model).

In [ ]:
_name = "nvidia"

if _name not in PROVIDER_OBJS:
    print(f"[{_name}] not available — skipped (see § 5 smoke test)")
else:
    _p = PROVIDER_OBJS[_name]
    _policy_a = make_policy_llm(_p, n_buildings=3, agent_label="α", timeout_s=LLM_TIMEOUT_S)
    _policy_b = make_policy_llm(_p, n_buildings=3, agent_label="β", timeout_s=LLM_TIMEOUT_S)

    print(f"=== Dual-agent: {_p.label} | timeout={LLM_TIMEOUT_S}s ===")
    _t0  = time.time()
    _run = run_policy_dual_agent(f"llm_{_name}", _policy_a, _policy_b)
    print(f"  wall clock: {time.time()-_t0:.0f}s")
    llm_runs[_name] = {"label": _p.label, **_run}
    print(f"  saved to llm_runs['{_name}']. Active runs: {list(llm_runs.keys())}")

## § 13 — Google Gemma: `gemma-3-12b-it`

> **Independent cell.** Interrupt safely if it hangs.  
> Uses Google AI Studio's OpenAI-compatible endpoint (`https://generativelanguage.googleapis.com/v1beta/openai/`).  
> Requires `GOOGLE_API_KEY` from [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (free tier available).  
> To use a larger model swap to `gemma-3-27b-it` in `PROVIDERS` (§ 0).  
> Estimated cost: **free** on AI Studio free tier; minimal on pay-as-you-go.

In [ ]:
_name = "gemma"

if _name not in PROVIDER_OBJS:
    print(f"[{_name}] not available — skipped (see § 5 smoke test)")
else:
    _p = PROVIDER_OBJS[_name]
    _policy_a = make_policy_llm(_p, n_buildings=3, agent_label="α", timeout_s=LLM_TIMEOUT_S)
    _policy_b = make_policy_llm(_p, n_buildings=3, agent_label="β", timeout_s=LLM_TIMEOUT_S)

    print(f"=== Dual-agent: {_p.label} | timeout={LLM_TIMEOUT_S}s ===")
    _t0  = time.time()
    _run = run_policy_dual_agent(f"llm_{_name}", _policy_a, _policy_b)
    print(f"  wall clock: {time.time()-_t0:.0f}s")
    llm_runs[_name] = {"label": _p.label, **_run}
    print(f"  saved to llm_runs['{_name}']. Active runs: {list(llm_runs.keys())}")

## § 14 — District-Level Results

Only providers present in `llm_runs` appear here — skipped or interrupted providers
are silently omitted. Re-run the provider cell above to add a missing run without restarting.

In [ ]:
def summarize_district(df: pd.DataFrame, label: str) -> dict:
    net_cols = [f"net{i}" for i in range(6)]
    dist_net = df[net_cols].sum(axis=1)
    return {
        "policy":         label,
        "total_reward":   float(df["reward_sum"].sum()),
        "total_cost_est": float((dist_net * df["price"]).sum()),
        "peak_net_kW":    float(dist_net.max()),
        "total_net_kWh":  float(dist_net.sum()),
    }


rows = [
    summarize_district(df_noop,   "noop"),
    summarize_district(df_random, "random"),
    summarize_district(df_rbc,    "rbc"),
]
for name, run in llm_runs.items():
    rows.append(summarize_district(run["df"], f"llm_{name} (dual-agent)"))

summary = pd.DataFrame(rows).set_index("policy")
print("District summary — higher total_reward is better (less negative)")
display(summary.round(4))

In [ ]:
HEADLINE_KPIS = [
    "electricity_consumption_total",
    "cost_total",
    "carbon_emissions_total",
    "daily_peak_average",
    "ramping_average",
    "daily_one_minus_load_factor_average",
]


def district_kpis(env_obj) -> pd.Series:
    df = env_obj.evaluate()
    if "level" in df.columns:
        mask = df["level"].astype(str).str.lower() == "district"
    elif "name" in df.columns:
        mask = df["name"].astype(str).str.lower() == "district"
    else:
        mask = pd.Series(True, index=df.index)
    d = df[mask]
    if d.empty:
        d = df.groupby("cost_function", as_index=False)["value"].mean()
    return d.set_index("cost_function")["value"].astype(float)


kpi_cols = {
    "No Control":  district_kpis(env_noop),
    "Random":      district_kpis(env_random),
    "RBC":         district_kpis(env_rbc),
}
for name, run in llm_runs.items():
    kpi_cols[f"{run['label']}\n(dual-agent)"] = district_kpis(run["env"])

kpi_table = pd.DataFrame(kpi_cols).sort_index().round(4)
present   = [k for k in HEADLINE_KPIS if k in kpi_table.index]

print("District KPIs — lower is better; No Control ≈ 1.000")
display(kpi_table.loc[present] if present else kpi_table)

In [ ]:
if present:
    ax = kpi_table.loc[present].plot(kind="bar", figsize=(14, 5), width=0.78)
    ax.axhline(1.0, color="k", ls="--", lw=0.9, label="No Control = 1.0")
    ax.set_ylabel("KPI (lower is better)")
    ax.set_title(
        f"District KPIs — dual-agent LLM vs baselines  "
        f"(t={WEEK_START}..{WEEK_START+WEEK_LEN-1}, {WEEK_LEN} steps)"
    )
    ax.legend(loc="upper right", fontsize=7, ncol=2)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## § 15 — Per-Agent Breakdown (α vs β)

How much of the district reward does each agent contribute?  
Significant α/β imbalance means one sub-district is being poorly controlled.

In [ ]:
def per_agent_summary(
    df: pd.DataFrame,
    agent_name: str,
    bldg_indices: list[int],
) -> dict:
    """Compute per-agent metrics from the full 6-building rollout dataframe."""
    net_cols = [f"net{i}" for i in bldg_indices]
    r_cols   = [f"r{i}"   for i in bldg_indices]
    soc_cols = [f"soc{i}" for i in bldg_indices]
    a_cols   = [f"a{i}"   for i in bldg_indices]
    dist_net = df[net_cols].sum(axis=1)
    return {
        "agent":             agent_name,
        "buildings":         str(bldg_indices),
        "total_reward":      float(df[r_cols].sum().sum()),
        "mean_soc_pct":      float(df[soc_cols].mean().mean() * 100),
        "peak_net_kW":       float(dist_net.max()),
        "total_net_kWh":     float(dist_net.sum()),
        "mean_action":       float(df[a_cols].values.mean()),
        "std_action":        float(df[a_cols].values.std()),
    }


if llm_runs:
    agent_rows = []
    for name, run in llm_runs.items():
        agent_rows.append(per_agent_summary(run["df"], f"{name} / α", AGENT_A_BUILDINGS))
        agent_rows.append(per_agent_summary(run["df"], f"{name} / β", AGENT_B_BUILDINGS))
    agent_df = pd.DataFrame(agent_rows).set_index("agent")
    print("Per-agent breakdown — α (B0-2) vs β (B3-5)")
    display(agent_df.round(4))
else:
    print("No LLM runs in llm_runs yet — run provider cells above.")

In [ ]:
if llm_runs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    providers  = list(llm_runs.keys())
    reward_a   = [llm_runs[n]["df"]["reward_a"].sum() for n in providers]
    reward_b   = [llm_runs[n]["df"]["reward_b"].sum() for n in providers]
    x          = range(len(providers))

    # Panel 1 — reward split α vs β
    w = 0.35
    axes[0].bar([i - w/2 for i in x], reward_a, width=w, label="Agent α (B0-2)", alpha=0.85, color="steelblue")
    axes[0].bar([i + w/2 for i in x], reward_b, width=w, label="Agent β (B3-5)", alpha=0.85, color="darkorange")
    axes[0].set_xticks(list(x)); axes[0].set_xticklabels(providers)
    axes[0].set_ylabel("Total reward (higher = better)")
    axes[0].set_title("Reward split: α vs β per provider")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, axis="y")

    # Panel 2 — fallback counts
    fb_a = [sum(1 for r in llm_runs[n]["raw_log_a"] if r["fallback"]) for n in providers]
    fb_b = [sum(1 for r in llm_runs[n]["raw_log_b"] if r["fallback"]) for n in providers]
    axes[1].bar([i - w/2 for i in x], fb_a, width=w, label="Agent α fallbacks", alpha=0.85, color="steelblue")
    axes[1].bar([i + w/2 for i in x], fb_b, width=w, label="Agent β fallbacks", alpha=0.85, color="darkorange")
    axes[1].set_xticks(list(x)); axes[1].set_xticklabels(providers)
    axes[1].set_ylabel("Fallback count (timeout / parse failure)")
    axes[1].set_title("Fallbacks per agent (target: 0)")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

## § 16 — Diagnostics

In [ ]:
# 15.1  SoC trajectories — coloured by agent group, one column per provider
if not llm_runs:
    print("No LLM runs yet.")
else:
    panels = [(df_rbc, "RBC (ref)")] + [(run["df"], run["label"]) for run in llm_runs.values()]
    fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5), sharey=True)
    if len(panels) == 1:
        axes = [axes]

    colors_a = ["#1f77b4", "#aec7e8", "#6baed6"]   # blues for α
    colors_b = ["#d62728", "#ff9896", "#e6550d"]   # reds  for β

    for ax, (df_, label) in zip(axes, panels):
        for local_i, global_i in enumerate(AGENT_A_BUILDINGS):
            ax.plot(df_["t"], df_[f"soc{global_i}"] * 100,
                    label=f"α-B{global_i}", lw=1.4, color=colors_a[local_i])
        for local_i, global_i in enumerate(AGENT_B_BUILDINGS):
            ax.plot(df_["t"], df_[f"soc{global_i}"] * 100,
                    label=f"β-B{global_i}", lw=1.4, color=colors_b[local_i], ls="--")

        # Shade PEAK price hours
        peak_mask = (df_["price"] >= PRICE_PEAK_THRESHOLD).values
        in_peak, span_s = False, 0
        for i_m, is_peak in enumerate(peak_mask):
            if is_peak and not in_peak:
                span_s = i_m; in_peak = True
            elif not is_peak and in_peak:
                ax.axvspan(span_s, i_m - 1, color="gold", alpha=0.12); in_peak = False
        if in_peak:
            ax.axvspan(span_s, len(peak_mask) - 1, color="gold", alpha=0.12)

        ax.set_xlabel("hour t"); ax.set_ylabel("SoC (%)")
        ax.set_title(label, fontsize=9)
        ax.legend(ncol=2, fontsize=6.5); ax.grid(alpha=0.3)

    plt.suptitle("SoC trajectories — blue=Agent α (B0-2), red=Agent β (B3-5) | gold=PEAK price", fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
# 15.2  District net load comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

def _plot_net(ax, dfs_labels, title):
    colors = ["gray", "steelblue", "darkorange", "crimson", "forestgreen", "purple"]
    for (df_, lbl), color in zip(dfs_labels, colors):
        net = df_[[f"net{i}" for i in range(6)]].sum(axis=1)
        ax.plot(df_["t"], net, label=lbl, lw=1.4, color=color, alpha=0.85)
    ax.set_xlabel("hour t"); ax.set_ylabel("district net load (kWh)")
    ax.set_title(title); ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)

ref_panels = [(df_noop, "No Control"), (df_random, "Random"), (df_rbc, "RBC")]
_plot_net(axes[0], ref_panels, "Reference baselines")

llm_panels = [(run["df"], run["label"]) for run in llm_runs.values()]
if llm_panels:
    _plot_net(axes[1], llm_panels, "Dual-agent LLM providers")
else:
    axes[1].text(0.5, 0.5, "No LLM runs yet", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# 15.3  Behavioural summary per provider (synchronisation, rule violations)
def analyze_dual_run(run: dict) -> list[dict]:
    rows = []
    df   = run["df"]
    for agent_lbl, bldg_ids, raw_log in [
        ("α", AGENT_A_BUILDINGS, run["raw_log_a"]),
        ("β", AGENT_B_BUILDINGS, run["raw_log_b"]),
    ]:
        a_cols   = [f"a{i}" for i in bldg_ids]
        soc_cols = [f"soc{i}" for i in bldg_ids]
        A        = df[a_cols].values
        signs    = np.sign(A)
        sync     = float(np.all(signs == signs[:, [0]], axis=1).mean())
        fb       = float(np.mean([r["fallback"] for r in raw_log])) if raw_log else 0.0
        SOC_post = df[soc_cols].values
        SOC_pre  = np.vstack([np.zeros(len(bldg_ids)), SOC_post[:-1]])
        rows.append({
            "provider":          run["label"],
            "agent":             agent_lbl,
            "sync_rate_pct":     round(sync * 100, 1),
            "fallback_pct":      round(fb * 100, 1),
            "charge_at_full":    int(((A > 0) & (SOC_pre >= 0.9)).sum()),
            "discharge_at_empty": int(((A < 0) & (SOC_pre <= 0.1)).sum()),
            "mean_action":       round(float(A.mean()), 3),
            "std_action":        round(float(A.std()), 3),
        })
    return rows


if llm_runs:
    beh_rows = []
    for run in llm_runs.values():
        beh_rows.extend(analyze_dual_run(run))
    beh_df = pd.DataFrame(beh_rows)
    print("Behavioural summary:")
    display(beh_df.set_index(["provider", "agent"]).round(3))
    print(
        "\nLow sync_rate is GOOD (agents act independently → lower demand spikes).\n"
        "fallback_pct = timeout + parse failures (target: 0).\n"
        "charge_at_full / discharge_at_empty = prompt-rule violations."
    )

In [ ]:
# 15.4  Sample raw responses — 1 timestep per provider, both agents side by side
N_SHOW = 1

if not llm_runs:
    print("No LLM runs yet.")
else:
    rng_diag = np.random.default_rng(SEED)
    ref_log  = next(iter(llm_runs.values()))["raw_log_a"]
    picked   = sorted(rng_diag.choice(len(ref_log), size=min(N_SHOW, len(ref_log)), replace=False).tolist())

    for idx in picked:
        t_val = ref_log[idx]["t"]
        print("═" * 72)
        print(f"t={t_val}")
        for name, run in llm_runs.items():
            entry_a = run["raw_log_a"][idx]
            entry_b = run["raw_log_b"][idx]
            print(f"\n── {run['label']} ──")
            print(f"  Agent α state:\n{entry_a['state_text']}")
            print(f"  Agent α response (fallback={entry_a['fallback']}):\n{entry_a['raw']}")
            print(f"  Agent β state:\n{entry_b['state_text']}")
            print(f"  Agent β response (fallback={entry_b['fallback']}):\n{entry_b['raw']}")
        print()

## § 17 — Save Artifacts

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M%S")

# ── Rollout CSVs ──────────────────────────────────────────────────────────
all_dfs = [df_noop, df_random, df_rbc] + [run["df"] for run in llm_runs.values()]
df_all  = pd.concat(all_dfs, ignore_index=True)
csv_path = ARTIFACTS / f"02_rollouts_{stamp}.csv"
df_all.to_csv(csv_path, index=False)
print(f"Rollout data    : {csv_path}")

# ── KPI table ─────────────────────────────────────────────────────────────
if present:
    kpi_path = ARTIFACTS / f"02_kpis_{stamp}.csv"
    kpi_table.loc[present].to_csv(kpi_path)
    print(f"KPI table       : {kpi_path}")

# ── Per-agent summary ─────────────────────────────────────────────────────
if llm_runs:
    agent_path = ARTIFACTS / f"02_per_agent_{stamp}.csv"
    agent_df.to_csv(agent_path)
    print(f"Per-agent table : {agent_path}")

# ── Raw LLM logs (JSON) ───────────────────────────────────────────────────
for name, run in llm_runs.items():
    for agent_key in ("raw_log_a", "raw_log_b"):
        tag      = "alpha" if agent_key == "raw_log_a" else "beta"
        log_path = ARTIFACTS / f"02_raw_{name}_{tag}_{stamp}.json"
        with open(log_path, "w") as f:
            json.dump(run[agent_key], f, indent=2)
        print(f"Raw log {name}/{tag:5s}: {log_path}")